# **Bank Customer Churn Prediction**

In [ ]:
import numpy as np
import pandas as pd

churn_df = pd.read_csv('bank.data.csv')
churn_df.head()

# Part 1: Data Exploration

In [ ]:
# check data info
churn_df.info()

In [ ]:
#check the unique value
churn_df.nunique()

In [ ]:
y = churn_df['Exited']

In [ ]:
# check missing values
churn_df.isnull().sum()

**There are no missing values in the data.**

In [ ]:
#check the description of the meaningful data
churn_df[['CreditScore', 'Age', 'Tenure', 'NumOfProducts','Balance', 'EstimatedSalary']].describe()

**Before making a formal prediction of customer churn, it is necessary to find what factors are related to customer churn.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

_,axss = plt.subplots(2,3, figsize=[20,10])
sns.boxplot(x='Exited', y ='CreditScore', data=churn_df, ax=axss[0][0])
sns.boxplot(x='Exited', y ='Age', data=churn_df, ax=axss[0][1])
sns.boxplot(x='Exited', y ='Tenure', data=churn_df, ax=axss[0][2])
sns.boxplot(x='Exited', y ='NumOfProducts', data=churn_df, ax=axss[1][0])
sns.boxplot(x='Exited', y ='Balance', data=churn_df, ax=axss[1][1])
sns.boxplot(x='Exited', y ='EstimatedSalary', data=churn_df, ax=axss[1][2])

Relatively speaking, the difference between customer churn and credit score, numbers of products, and estimated salary is not significant. For age, users with existing=1 are generally older, which is reflected in a larger average value; For tenure, the difference between the maximum and minimum values of tenure for churn users is greater than that for non-churn users; For balance, the average value of lost users is larger than that of non-lost users, and the lower limit of balance is also higher.

In [ ]:
_,axss = plt.subplots(2,2, figsize=[20,10])
sns.countplot(x='Exited', hue='Geography', data=churn_df, ax=axss[0][0])
sns.countplot(x='Exited', hue='Gender', data=churn_df, ax=axss[0][1])
sns.countplot(x='Exited', hue='HasCrCard', data=churn_df, ax=axss[1][0])
sns.countplot(x='Exited', hue='IsActiveMember', data=churn_df, ax=axss[1][1])

As shown in the figure, geography has relatively large data differences, which is reflected in the significant decrease in the proportion of French users and the significant increase in the proportion of Germans among the lost users. In terms of gender, unlike the situation where there are more men than women who have not lost customers, there is a phenomenon of more women than men among lost customers. Among the non-lost customers, there are more active members, while among the lost customers, the opposite is true. Overall, the difference in HasCrCard factor between non-lost and lost customers is not significant.

# Part 2: Feature Preprocessing

In [ ]:
#firstly,drop the useless data
to_drop = ['RowNumber','CustomerId','Surname','Exited']
X = churn_df.drop(to_drop, axis = 1)
X.head()

In [ ]:
X.dtypes

**There are two types of data in the data that are 'object', so it is necessary to encode and process these two types of data. In order to prevent the early leakage of test data, we need to split the data in advance**

In [ ]:
from sklearn import model_selection

# take the 20% of data as the test data
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, stratify = y, random_state = 100)

print('training data has ' + str(X_train.shape[0]) + ' observation with ' + str(X_train.shape[1]) + ' features')
print('test data has ' + str(X_test.shape[0]) + ' observation with ' + str(X_test.shape[1]) + ' features')

In [ ]:
cat_cols = X.columns[X.dtypes == 'object']
num_cols = X.columns[(X.dtypes == 'float64') | (X.dtypes == 'int64')]
X_train.head()

In [ ]:
# One hot encoding
from sklearn.preprocessing import OneHotEncoder

def OneHotEncoding(df, enc, categories):
  transformed = pd.DataFrame(enc.transform(df[categories]).toarray(), columns = enc.get_feature_names_out(categories))
  return pd.concat([df.reset_index(drop=True), transformed], axis=1).drop(categories, axis=1)

categories = ['Geography']
enc_ohe = OneHotEncoder()
enc_ohe.fit(X_train[categories])

X_train = OneHotEncoding(X_train, enc_ohe, categories)
X_test = OneHotEncoding(X_test, enc_ohe, categories)

In [ ]:
X_train.head()

In [ ]:
# Ordinal encoding
from sklearn.preprocessing import OrdinalEncoder

categories = ['Gender']
enc_oe = OrdinalEncoder()
enc_oe.fit(X_train[categories])

X_train[categories] = enc_oe.transform(X_train[categories])
X_test[categories] = enc_oe.transform(X_test[categories])

In [ ]:
X_train.head()

In [ ]:
#Standard
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train[num_cols])

X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
X_train.head()

# Part 3: Model Training and Result Evaluation

In [ ]:
#build models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression
classifier_logistic = LogisticRegression()

# K Nearest Neighbors
classifier_KNN = KNeighborsClassifier()

# Random Forest
classifier_RF = RandomForestClassifier()

In [ ]:
# Train the model
classifier_logistic.fit(X_train, y_train)
classifier_RF.fit(X_train,y_train)
classifier_KNN.fit(X_train,y_train)

In [ ]:
# Prediction of logistic regression
classifier_logistic.predict(X_test)

In [ ]:
# Prediction of KNN
classifier_KNN.predict(X_test)

In [ ]:
# Prediction of random forest
classifier_RF.predict(X_test)

**In order to prevent overfitting as well as find the best model,we need to find a better parameter**

In [ ]:
from sklearn.model_selection import GridSearchCV

# helper function for printing out grid search results
def print_grid_search_metrics(gs):
    print ("Best score: " + str(gs.best_score_))
    print ("Best parameters set:")
    best_parameters = gs.best_params_
    for param_name in sorted(best_parameters.keys()):
        print(param_name + ':' + str(best_parameters[param_name]))

In [ ]:
#Logistic Regression（Regularization）
parameters = {
    'penalty':('l2','l1'),
    'C':(0.01, 0.05, 0.1, 0.2, 1)
}

Grid_LR = GridSearchCV(LogisticRegression(solver='liblinear'),parameters, cv = 5)
Grid_LR.fit(X_train, y_train)
print_grid_search_metrics(Grid_LR)

In [ ]:
best_LR_model = Grid_LR.best_estimator_

In [ ]:
best_LR_model.predict(X_test)

In [ ]:
best_LR_model.score(X_test, y_test)

In [ ]:
LR_models = pd.DataFrame(Grid_LR.cv_results_)
res = (LR_models.pivot(index='param_penalty', columns='param_C', values='mean_test_score'))
_ = sns.heatmap(res, cmap='viridis')

It can be seen that there is a difference in the performance of different C values under L1 and L2, with the best performance at l1 and C=0.01, which is 0.802 as mentioned above

In [ ]:
#KNN
parameters = {
    'n_neighbors':[1,3,5,7,9]
}
Grid_KNN = GridSearchCV(KNeighborsClassifier(),parameters, cv=5)
Grid_KNN.fit(X_train, y_train)
print_grid_search_metrics(Grid_KNN)

In [ ]:
best_KNN_model = Grid_KNN.best_estimator_
best_KNN_model.predict(X_test)

In [ ]:
best_KNN_model.score(X_test, y_test)

In [ ]:
#Random Forest
parameters = {
    'n_estimators' : [60,80,100],
    'max_depth': [1,5,10]
}
Grid_RF = GridSearchCV(RandomForestClassifier(),parameters, cv=5)
Grid_RF.fit(X_train, y_train)
print_grid_search_metrics(Grid_RF)

In [ ]:
best_RF_model = Grid_RF.best_estimator_
best_RF_model.score(X_test, y_test)

In [ ]:
Prediction_logistic = best_LR_model.predict(X_test)
Prediction_KNN = best_KNN_model.predict(X_test)
Prediction_RF = best_RF_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, Prediction_logistic))

In [ ]:
print(classification_report(y_test, Prediction_KNN))

In [ ]:
print(classification_report(y_test, Prediction_RF))

Overall, in terms of precision, recall, accuracy, and score, Random Forest performs the best among the three models.

### ROC curve

In [ ]:
#RF
from sklearn.metrics import roc_curve
from sklearn import metrics

# Use predict_proba to get the probability results of Random Forest
y_pred_rf = best_RF_model.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_rf)

In [ ]:
best_RF_model.predict_proba(X_test)

In [ ]:
# ROC curve of Random Forest result
import matplotlib.pyplot as plt
plt.figure(1)
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_rf, tpr_rf, label='RF')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve - RF model')
plt.legend(loc='best')
plt.show()

In [ ]:
from sklearn import metrics

# AUC score
AUC_RF = metrics.auc(fpr_rf,tpr_rf)
AUC_RF

In [ ]:
#LG
y_pred_lr = best_LR_model.predict_proba(X_test)[:, 1]
fpr_lr, tpr_lr, thresh = roc_curve(y_test, y_pred_lr)

In [ ]:
best_LR_model.predict_proba(X_test)

In [ ]:
# ROC Curve
plt.figure(1)
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_lr, tpr_lr, label='LR')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve - LR Model')
plt.legend(loc='best')
plt.show()

In [ ]:
# AUC score
AUC_LR = metrics.auc(fpr_lr,tpr_lr)
AUC_LR

In [ ]:
#KNN
y_pred_knn = best_KNN_model.predict_proba(X_test)[:, 1]
fpr_knn, tpr_knn,thresh_ = roc_curve(y_test, y_pred_knn)

In [ ]:
best_KNN_model.predict_proba(X_test)

In [ ]:
# ROC Curve
plt.figure(1)
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_knn, tpr_knn, label='KNN')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve - KNN Model')
plt.legend(loc='best')
plt.show()

In [ ]:
# AUC score
AUC_KNN = metrics.auc(fpr_knn,tpr_knn)
AUC_KNN

In [ ]:
#comparison
from sklearn.metrics import roc_curve, auc
plt.figure(1)

plt.plot([0, 1], [0, 1], 'k--')

plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {AUC_RF:.3f})', linewidth=2)
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {AUC_LR:.3f})', linewidth=2)
plt.plot(fpr_knn, tpr_knn, label=f'KNN (AUC = {AUC_KNN:.3f})', linewidth=2)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison', fontsize=14)
plt.legend(loc='best')

plt.show()

Although the LR model performs better than KNN at lower thresholds,RF performs better than KNN overall, and both are better than LR. This can also be seen from AUC.

# Part 4: Model Extra Functionality

In [ ]:
X_with_corr = X.copy()

X_with_corr = OneHotEncoding(X_with_corr, enc_ohe, ['Geography'])
X_with_corr['Gender'] = enc_oe.transform(X_with_corr[['Gender']])
X_with_corr['SalaryInRMB'] = X_with_corr['EstimatedSalary'] * 6.4
X_with_corr.head()

In [ ]:
X_RF = X.copy()

X_RF = OneHotEncoding(X_RF, enc_ohe, ['Geography'])
X_RF['Gender'] = enc_oe.transform(X_RF[['Gender']])

X_RF.head()

**It is not difficult to find from the above analysis that Random Forest performs the best among the three models, so we will first rank this model based on feature importance**

In [ ]:
forest = RandomForestClassifier()
forest.fit(X_RF, y)

importances = forest.feature_importances_

indices = np.argsort(importances)[::-1]

# Print the feature ranking
print("Feature importance ranking by Random Forest Model:")
for ind in range(X.shape[1]):
  print ("{0} : {1}".format(X_RF.columns[indices[ind]],round(importances[indices[ind]], 4)))

In [ ]:
# draw a pie
plt.figure(figsize=(10, 8))
features = X_RF.columns[indices]
importance_values = importances[indices]

threshold = 0.02
significant_features = features[importance_values > threshold]
significant_importances = importance_values[importance_values > threshold]
other_importance = sum(importance_values[importance_values <= threshold])

if other_importance > 0:
    significant_features = np.append(significant_features, 'Others')
    significant_importances = np.append(significant_importances, other_importance)

colors = plt.cm.tab20c(np.linspace(0, 1, len(significant_importances)))

wedges, texts, autotexts = plt.pie(
    significant_importances,
    labels=significant_features,
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.85,
    wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
    textprops={'fontsize': 10}
)

plt.title('Feature Importance from Random Forest Model', fontsize=16, pad=20)

plt.setp(autotexts, size=10, weight="bold")
plt.setp(texts, size=10)

plt.legend(
    wedges,
    significant_features,
    title="Features",
    loc="center left",
    bbox_to_anchor=(1, 0.5, 0.5, 1),
    fontsize=10
)

plt.axis('equal')
plt.tight_layout()
plt.show()

It can be seen that age has the greatest impact, followed by estimated salary, credit score, balance, num of products, and tenure, which is roughly consistent with the initial results of EDA.

# Part 5:Practical Business

#### This is a simple practice.The main function is to first store the data of a batch of new customers in today. csv, predict whether this batch of new customers will be lost through the model, and finally store the predicted customers who may be lost in warning. csv, so that the bank can carry out subsequent retention work for these customers based on the information in the warning.

In [ ]:
#clean the warning.csv
warning = pd.DataFrame()
warning.to_csv('warning.csv')

In [ ]:
#choose to use Random Forest
churn_df_new = pd.read_csv('today.csv')
X_new = churn_df_new.drop(to_drop, axis = 1)
y_new = churn_df_new['Exited']
churn_df_new.head()

In [ ]:
X_new = OneHotEncoding(X_new, enc_ohe, ['Geography'])
X_new[categories]=enc_oe.transform(X_new[categories])
X_new.head()

In [ ]:
print('result of RandomForest：',classifier_RF.predict(X_new))

In [ ]:
churn_df_new['Exited'] = classifier_RF.predict(X_new)

warning = pd.concat([warning,churn_df_new[churn_df_new['Exited']==0].drop(['Exited','RowNumber'],axis=1)])
warning.to_csv('warning.csv',index= False)
warning